# CA-13 Housing Vulnerability Microsimulation
## Complete Pipeline Demonstration

**From Shelter Ratios to Residual Income: A Disaggregate Analysis of Housing Hardship and Macroeconomic Shocks in Canada**

---

This notebook demonstrates the complete CA-13 microsimulation pipeline:

1. **Data Loading**: Census 2021 PUMF baseline
2. **Asset Imputation**: Statistical matching with SFS 2023
3. **Shock Scenarios**: 
   - Trade War (Canada-USA)
   - AI Unemployment
   - Immigration Rent Surge
   - Mortgage Rate Shock
   - Compound Shocks
4. **Vulnerability Analysis**: Months-to-default, cohort impacts
5. **Policy Counterfactuals**: Universal vs targeted transfers

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# CA-13 modules
from data_loading import CensusLoader, SFSLoader
from imputation import AssetImputer
from shocks import TradeWarShock, AIUnemploymentShock

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ CA-13 modules loaded successfully")

## Step 1: Load Census 2021 PUMF Baseline

Load ~930K household records and calculate baseline metrics:
- STIR (Shelter-Cost-to-Income Ratio)
- Residual Income
- Tenure status

In [ ]:
# Initialize Census loader
census_loader = CensusLoader(config_path='../config/data_paths.yaml')

# For this demo, create synthetic data (replace with actual Census PUMF when available)
print("Creating synthetic household data for demonstration...")
print("(Replace with actual Census PUMF: census_loader.load_census_pumf('data/raw/census_2021_hier_v2.csv'))")

n_households = 10000  # Use 10K for demo (actual: ~930K)
np.random.seed(42)

# Create synthetic baseline
baseline_df = pd.DataFrame({'_id': range(n_households)})
baseline_df = census_loader._create_synthetic_baseline(baseline_df)
baseline_df = census_loader.calculate_baseline_metrics(baseline_df)

print(f"\n✓ Loaded {len(baseline_df):,} households")
print(f"  Mean annual income: ${baseline_df['annual_income'].mean():,.0f}")
print(f"  Mean STIR: {baseline_df['STIR'].mean():.1%}")
print(f"  % cost-burdened (>30%): {(baseline_df['STIR'] > 0.30).mean():.1%}")

### Visualize Baseline Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Income distribution
axes[0].hist(baseline_df['annual_income'] / 1000, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Annual Income ($1000s)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Household Income Distribution')
axes[0].axvline(baseline_df['annual_income'].median() / 1000, color='red', linestyle='--', label='Median')
axes[0].legend()

# STIR distribution
axes[1].hist(baseline_df['STIR'].clip(0, 1), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Shelter-Cost-to-Income Ratio (STIR)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('STIR Distribution')
axes[1].axvline(0.30, color='red', linestyle='--', label='30% threshold')
axes[1].legend()

# Residual income
axes[2].hist(baseline_df['residual_income_baseline'] / 1000, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[2].set_xlabel('Residual Income ($1000s)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Residual Income Distribution')
axes[2].axvline(0, color='red', linestyle='--', label='Zero threshold')
axes[2].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/baseline_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Baseline visualizations saved")

## Step 2: Statistical Asset Imputation

Impute liquid assets from SFS 2023 to estimate household liquidity buffers

In [ ]:
# Load SFS data (synthetic for demo)
sfs_loader = SFSLoader()
sfs_data = sfs_loader._create_synthetic_sfs(n=3000)

print("✓ SFS data loaded")
print(f"  {len(sfs_data):,} households with asset data")
print(f"  Mean liquid assets: ${sfs_data['liquid_assets'].mean():,.0f}")

# Impute assets onto Census households
imputer = AssetImputer(sfs_data, method='hot_deck')
enriched_df = imputer.impute_assets(baseline_df)

print("\n✓ Asset imputation complete")
print(f"  Mean imputed liquid assets: ${enriched_df['liquid_assets_imputed'].mean():,.0f}")
print(f"  Households with <$5K: {(enriched_df['liquid_assets_imputed'] < 5000).sum():,}")

## Step 3: Shock Scenario Analysis

### Scenario 1: Trade War (Canada-USA)

In [ ]:
# Apply moderate trade war shock
trade_shock = TradeWarShock(severity='moderate')
trade_shocked = trade_shock.apply_shock(enriched_df.copy())

# Calculate household-level impacts
trade_shocked['income_loss_pct'] = (
    (trade_shocked['annual_income'] - trade_shocked['income_shocked']) / 
    trade_shocked['annual_income']
)

# Summary by industry
industry_impact = trade_shocked.groupby('industry').agg({
    'trade_job_lost': 'mean',
    'income_loss_pct': 'mean',
    'industry': 'count'
}).rename(columns={'industry': 'n_households'})

print("\n=== Trade War Impact by Industry ===")
print(industry_impact.sort_values('trade_job_lost', ascending=False))

### Scenario 2: AI Unemployment

In [ ]:
# Apply 5-year AI unemployment shock
ai_shock = AIUnemploymentShock(timeline='5_year')
ai_shocked = ai_shock.apply_shock(enriched_df.copy())

# Occupation-level impacts
occupation_impact = ai_shocked.groupby('occupation').agg({
    'ai_job_lost': 'mean',
    'ai_displacement_risk': 'mean',
    'occupation': 'count'
}).rename(columns={'occupation': 'n_households'})

print("\n=== AI Unemployment Impact by Occupation ===")
print(occupation_impact.sort_values('ai_job_lost', ascending=False).head(10))

### Compare Shock Impacts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trade war impact
axes[0].barh(range(len(industry_impact)), industry_impact['trade_job_lost'] * 100)
axes[0].set_yticks(range(len(industry_impact)))
axes[0].set_yticklabels(industry_impact.index)
axes[0].set_xlabel('Job Loss Rate (%)')
axes[0].set_title('Trade War: Job Losses by Industry')
axes[0].grid(axis='x', alpha=0.3)

# AI impact (top 10 occupations)
top_ai = occupation_impact.sort_values('ai_job_lost', ascending=False).head(10)
axes[1].barh(range(len(top_ai)), top_ai['ai_job_lost'] * 100, color='red', alpha=0.7)
axes[1].set_yticks(range(len(top_ai)))
axes[1].set_yticklabels(top_ai.index)
axes[1].set_xlabel('Job Loss Rate (%)')
axes[1].set_title('AI Unemployment: Top 10 At-Risk Occupations')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/shock_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 4: Liquidity Buffer Analysis

Calculate "months to default" for households in distress

In [ ]:
# Analyze liquidity under trade war scenario
in_distress = trade_shocked['residual_income_shocked'] < 0

print(f"\n=== Liquidity Buffer Analysis (Trade War) ===")
print(f"  Households in distress: {in_distress.sum():,} ({in_distress.mean():.1%})")

if in_distress.any():
    distressed = trade_shocked[in_distress].copy()
    
    # Calculate months until asset depletion
    monthly_deficit = distressed['residual_income_shocked'].abs() / 12
    months_covered = (distressed['liquid_assets_imputed'] / monthly_deficit).clip(0, 120)
    
    # Categorize
    print(f"\n  Liquidity categories:")
    print(f"    Immediate insolvency (<1 month): {(months_covered < 1).sum():,}")
    print(f"    Critical (1-3 months): {((months_covered >= 1) & (months_covered < 3)).sum():,}")
    print(f"    Vulnerable (3-6 months): {((months_covered >= 3) & (months_covered < 6)).sum():,}")
    print(f"    Moderate buffer (6-12 months): {((months_covered >= 6) & (months_covered < 12)).sum():,}")
    print(f"    Strong buffer (>12 months): {(months_covered >= 12).sum():,}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(months_covered.clip(0, 24), bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(3, color='red', linestyle='--', linewidth=2, label='3-month threshold')
    ax.axvline(6, color='orange', linestyle='--', linewidth=2, label='6-month threshold')
    ax.set_xlabel('Months Until Asset Depletion')
    ax.set_ylabel('Number of Households')
    ax.set_title('Liquidity Buffer Distribution (Households in Distress)')
    ax.legend()
    plt.savefig('../outputs/figures/liquidity_buffers.png', dpi=300, bbox_inches='tight')
    plt.show()

## Step 5: Policy Counterfactuals

Compare effectiveness of different intervention strategies

In [ ]:
# Policy scenarios
policies = {
    'Universal Housing Allowance': {
        'transfer': 500,  # $500/month to all households
        'targeted': False
    },
    'Targeted Low-Income Support': {
        'transfer': 800,  # $800/month to bottom income quintile
        'targeted': True,
        'income_threshold': 0.20  # Bottom 20%
    },
    'Emergency Job Loss Support': {
        'transfer': 1200,  # $1200/month to job losers only
        'targeted': 'job_loss'
    }
}

results = []

for policy_name, params in policies.items():
    df = trade_shocked.copy()
    
    # Apply transfer
    annual_transfer = params['transfer'] * 12
    
    if params['targeted'] == False:
        # Universal
        df['income_with_policy'] = df['income_shocked'] + annual_transfer
        eligible = len(df)
    elif params['targeted'] == True:
        # Income-targeted
        threshold = df['annual_income'].quantile(params['income_threshold'])
        eligible_mask = df['annual_income'] <= threshold
        df['income_with_policy'] = df['income_shocked'].copy()
        df.loc[eligible_mask, 'income_with_policy'] += annual_transfer
        eligible = eligible_mask.sum()
    else:
        # Job loss targeted
        df['income_with_policy'] = df['income_shocked'].copy()
        df.loc[df['trade_job_lost'], 'income_with_policy'] += annual_transfer
        eligible = df['trade_job_lost'].sum()
    
    # Recalculate residual income
    df['residual_with_policy'] = (df['income_with_policy'] * 0.75) - df['annual_shelter_cost']
    
    # Metrics
    baseline_distress = (df['residual_income_shocked'] < 0).sum()
    policy_distress = (df['residual_with_policy'] < 0).sum()
    households_saved = baseline_distress - policy_distress
    
    total_cost = eligible * annual_transfer
    cost_per_saved = total_cost / households_saved if households_saved > 0 else np.inf
    
    results.append({
        'Policy': policy_name,
        'Eligible HH': f"{eligible:,}",
        'HH Saved': f"{households_saved:,}",
        'Total Cost ($M)': f"${total_cost/1e6:,.1f}",
        'Cost per HH Saved': f"${cost_per_saved:,.0f}",
        'Efficiency Rank': 0  # Will fill after
    })

policy_df = pd.DataFrame(results)
print("\n=== Policy Counterfactual Comparison ===")
print(policy_df.to_string(index=False))

## Summary & Key Findings

This notebook demonstrated the complete CA-13 pipeline capabilities:

### Technical Implementation
1. ✓ Baseline microdata loading and metric calculation
2. ✓ Statistical asset imputation from SFS
3. ✓ Modular shock scenario framework
4. ✓ Liquidity buffer analysis
5. ✓ Policy counterfactual evaluation

### Key Advantages Over Traditional Models
- **Disaggregate Analysis**: Household-level impacts, not aggregate approximations
- **Residual Income Framework**: Captures true material deprivation vs simplistic 30% STIR
- **Asset Integration**: Distinguishes temporary cash-flow stress from insolvency
- **Flexible Scenarios**: Trade wars, AI unemployment, immigration, compound shocks
- **Policy Optimization**: Cost-effectiveness comparisons across interventions

### Next Steps
1. Replace synthetic data with actual Census 2021 PUMF (~930K households)
2. Implement full cohort stratification (7 vulnerable groups)
3. Add regional/CMA-specific analysis
4. Hindcast validation against 2023 CHS
5. Generate publication-ready figures for academic papers